# ML-02 — Research Question and Provisional Lane

This notebook defines the **Content Refresh Priority** lane, the operational decision it supports, and an initial empirical look at the 30,000-page FlyRank starter dataset.

## 1. My lane (or freestyle) and why

I chose the **Content Refresh Priority Prediction** lane.

The objective of this project is to identify website pages that should be reviewed first for a content refresh using only pre-decision search signals. This project combines interpretable machine learning with SEO analytics to help editorial teams prioritize content updates based on observed historical search performance.

## 2. The question: decision, action, cost of a wrong call

**Core Research Question:** *"Using only information available before the decision moment, which content items should be reviewed first for a possible refresh, and what signals explain that priority?"*

- **Decision:** Which website pages should be reviewed and refreshed first given limited weekly editorial capacity?
- **Action:** The SEO or content team audits and updates the top-ranked pages surfaced in the weekly refresh queue.
- **Cost of a Wrong Recommendation:** False positives waste writer and editor hours on pages that do not need updates or will not respond; false negatives leave high-exposure decaying pages unaddressed, forfeiting recoverable organic visibility.
- **Why Machine Learning:** Rule thresholds consider signals in isolation with rigid cutoffs, whereas an interpretable model weighs impressions, clicks, CTR, staleness, and ranking position simultaneously.

## 3. Quick look at the data (2-3 real numbers)

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root() -> Path:
    cur = Path.cwd().resolve()
    for p in [cur, *cur.parents]:
        if (p / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return p
    return cur

REPO_ROOT = find_repo_root()
df = pd.read_csv(REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].astype(str).str.lower() == "down").astype(int)

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Unique clients:", df["client_id"].nunique())
print("Positive declining base rate:", round(float(df["is_declining_label"].mean()), 4))

print("\nTrend Direction Distribution:")
print(df["trend_direction"].value_counts())

corr_val = df["search_volume"].corr(df["impressions_90d"])
print("\nCorrelation (search_volume vs impressions_90d):", round(float(corr_val), 6))


Rows: 30000
Columns: 45
Unique clients: 32
Positive declining base rate: 0.5421

Trend Direction Distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Correlation (search_volume vs impressions_90d): 0.001203


### Quick Look at the Data

- The starter dataset contains **30,000 pages** across **32 pseudonymized clients** with **44 columns**.
- The trend distribution shows **16,262 pages marked as `down` (54.21%)**, **5,962 `stable`**, **4,388 `up`**, **2,236 `new`**, and **1,152 `flat`**, confirming that declining performance is widespread.
- The correlation between `search_volume` and `impressions_90d` is **0.001203**, which is very close to zero. This suggests that search volume alone is not enough to estimate page performance, motivating a multi-signal model.

## 4. Careful words: what I can and can't claim

- **What we can claim (decision-support):** This project provides **observed**, **measured**, and **directional** decision-support by ranking pages whose pre-decision search signals are associated with content decline.
- **What we cannot claim:** This project cannot prove why Google ranks pages in a certain way or guarantee future rankings. The recommendations assist human decision-making rather than replacing it.

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`